<a href="https://colab.research.google.com/github/diademsamuel/7316/blob/main/Copy_of_superv_HillCountry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.2 MB/s eta 0:00:00


In [2]:
import ee
import geemap

In [3]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-diademsamuel')

To address the 'OSM tiler not having a referer' error, you can explicitly set a different basemap provider. Here's how to add a Google Satellite basemap:

In [10]:
Map = geemap.Map()
Map.add_basemap('USGS.USImagery')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [31]:
# Define the approximate bounding box for the Hill Country in Central Texas
# Coordinates: [west, south, east, north]
# This rectangle covers a broad area including parts of the Hill Country
hill_country = ee.Geometry.Rectangle([-100.5, 29.5, -97.5, 31.5])

# Add the geometry to the map
Map.addLayer(hill_country, {'color': 'FF0000'}, 'Texas Hill Country')

# Center the map on the new geometry
Map.centerObject(hill_country, 8)
Map

Map(bottom=13715.0, center=[30.50515596845637, -99], controls=(WidgetControl(options=['position', 'transparent…

In [43]:
point = hill_country

image = (
    ee.ImageCollection("LANDSAT/LC08/C02/T2_L2")
    .filterBounds(point)
    .filterDate("2018-01-01", "2023-12-31")
    .sort("CLOUD_COVER")
    .first()
    .select("SR_B[1-7]")
)

vis_params = {"min": 0, "max": 3000, "bands": ["SR_B5", "SR_B4", "SR_B3"]}

Map.centerObject(point, 8)
Map.addLayer(image, vis_params, "Landsat-8")

In [44]:
hill_country_centroid = hill_country.centroid()
Map.addLayer(hill_country_centroid, {'color': '0000FF'}, 'Hill Country Centroid')
Map.centerObject(hill_country_centroid, 8)
Map

Map(bottom=13874.0, center=[30.50515596845637, -99], controls=(WidgetControl(options=['position', 'transparent…

In [45]:
acquisition_date = ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo()
cloud_cover = image.get("CLOUD_COVER").getInfo()

print(f"Acquisition Date of selected Landsat 8 image: {acquisition_date}")
print(f"Cloud Cover of selected Landsat 8 image: {cloud_cover}%")

Acquisition Date of selected Landsat 8 image: 2019-02-28
Cloud Cover of selected Landsat 8 image: 0.94%


If you want to find specific dates or images with better coverage, you would typically need to iterate through the image collection and visually inspect the images or analyze their cloud cover properties over a broader period. The current approach automatically selects the clearest image within the specified date range.

In [46]:
ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo()

'2019-02-28'

In [47]:
image.get("CLOUD_COVER").getInfo()

0.94

In [48]:
roi_rectangle = hill_country
roi_point = hill_country_centroid

Map.addLayer(roi_rectangle, {'color': 'green'}, 'ROI Rectangle (Hill Country)')
Map.addLayer(roi_point, {'color': 'orange'}, 'ROI Point (Hill Country Centroid)')
Map

Map(bottom=13731.0, center=[30.831497881307943, -97.17383075609797], controls=(WidgetControl(options=['positio…

In [49]:
nlcd = ee.Image("USGS/NLCD/NLCD2016").select("landcover").clip(image.geometry())
Map.addLayer(nlcd, {}, "NLCD")
Map

Map(bottom=3694.0, center=[29.458731185355344, -93.20529641142343], controls=(WidgetControl(options=['position…

In [50]:
# Make the training dataset.
points = nlcd.sample(
    **{
        "region": image.geometry(),
        "scale": 30,
        "numPixels": 5000,
        "seed": 0,
        "geometries": True,  # Set this to False to ignore geometries
    }
)

Map.addLayer(points, {}, "training", False)

In [51]:
print(points.size().getInfo())

5000


In [52]:
print(points.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-99.72419470090709, 29.56550221234539]}, 'id': '0', 'properties': {'landcover': 52}}


In [53]:
# Use these bands for prediction.
bands = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]


# This property of the table stores the land cover labels.
label = "landcover"

# Overlay the points on the imagery to get training.
training = image.select(bands).sampleRegions(
    **{"collection": points, "properties": [label], "scale": 30}
)

# Train a CART classifier with default parameters.
trained = ee.Classifier.smileCart().train(training, label, bands)


In [54]:
print(training.first().getInfo())

{'type': 'Feature', 'geometry': None, 'id': '0_0', 'properties': {'SR_B1': 50860, 'SR_B2': 50721, 'SR_B3': 48287, 'SR_B4': 47810, 'SR_B5': 45581, 'SR_B6': 14170, 'SR_B7': 14560, 'landcover': 52}}


In [55]:
# Classify the image with the same bands used for training.
result = image.select(bands).classify(trained)

# # Display the clusters with random colors.
Map.addLayer(result.randomVisualizer(), {}, "classified")
Map

Map(bottom=3694.0, center=[29.458731185355344, -93.20529641142343], controls=(WidgetControl(options=['position…

In [56]:
class_values = nlcd.get("landcover_class_values").getInfo()
class_values

[11,
 12,
 21,
 22,
 23,
 24,
 31,
 41,
 42,
 43,
 51,
 52,
 71,
 72,
 73,
 74,
 81,
 82,
 90,
 95]

In [57]:
class_palette = nlcd.get("landcover_class_palette").getInfo()
class_palette

['476ba1',
 'd1defa',
 'decaca',
 'd99482',
 'ee0000',
 'ab0000',
 'b3aea3',
 '68ab63',
 '1c6330',
 'b5ca8f',
 'a68c30',
 'ccba7d',
 'e3e3c2',
 'caca78',
 '99c247',
 '78ae94',
 'dcd93d',
 'ab7028',
 'bad9eb',
 '70a3ba']

In [58]:
landcover = result.set("classification_class_values", class_values)
landcover = landcover.set("classification_class_palette", class_palette)

In [59]:
Map.addLayer(landcover, {}, "Land cover")
Map

Map(bottom=3694.0, center=[29.458731185355344, -93.20529641142343], controls=(WidgetControl(options=['position…

In [60]:
print("Change layer opacity:")
cluster_layer = Map.layers[-1]
cluster_layer.interact(opacity=(0, 1, 0.1))

Change layer opacity:


Box(children=(FloatSlider(value=1.0, description='opacity', max=1.0),))

In [61]:
Map.add_legend(builtin_legend="NLCD")
Map

Map(bottom=3694.0, center=[29.458731185355344, -93.20529641142343], controls=(WidgetControl(options=['position…

In [62]:
geemap.ee_export_image_to_drive(
    landcover, description="landcover", folder="export_HillCountryLandcover", scale=900
)